In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# 本脚本所在文件夹(也是输出目录)。Jupyter/Colab 里没有 __file__, 退回当前工作目录。
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
OUT = HERE

def find_raw(name: str) -> Path:
    """按顺序查找原始文件: 上一级目录 -> 本目录 -> 当前工作目录。
    这样本地(原始数据在上一级)和 Colab(全部文件在同一目录)都能直接用。"""
    for d in (HERE.parent, HERE, Path.cwd()):
        p = d / name
        if p.exists():
            return p
    raise FileNotFoundError(f"找不到原始文件: {name}")

# 研究窗口: 车身分类 2022-07 变更 -> 从该月开始; 数据最新到 2026-07
STUDY_START = pd.Timestamp("2022-07-01")
STUDY_END   = pd.Timestamp("2026-07-01")

log_rows = []          # 清洗日志
def log(item, result):
    log_rows.append({"check": item, "result": result})
    print(f"  {item}: {result}")

print("=" * 78)
print("BYD 数据清洗")
print("=" * 78)
print("脚本目录    :", HERE)
print("输出目录    :", OUT)
print()

BYD 数据清洗
脚本目录    : C:\Users\56426\Desktop\ABD\BYD EV\cleaned_data
输出目录    : C:\Users\56426\Desktop\ABD\BYD EV\cleaned_data



In [2]:
# 1. M03 月度注册 —— 分析主干(与同学的清洗口径一致)
# =============================================================================
print("[1] 清洗 M03-Car_Regn_by_make.csv")

m03_path = find_raw("M03-Car_Regn_by_make.csv")
source_hash = hashlib.sha256(m03_path.read_bytes()).hexdigest()

# 全部按字符串读入, 避免 pandas 对空值/编码的自动推断影响判断
raw = pd.read_csv(m03_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
raw.columns = raw.columns.str.strip()

required_columns = ["month", "make", "importer_type", "fuel_type", "vehicle_type", "number"]
missing_columns = [c for c in required_columns if c not in raw.columns]
if missing_columns:
    raise ValueError(f"M03 缺少字段: {missing_columns}")

df = raw[required_columns].copy()
for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

print(f"  原始行数: {len(raw)}  列: {raw.columns.tolist()}")
log("M03 原始行数", len(raw))
log("M03 源文件SHA256", source_hash[:16] + "...")

# --- 1.1 月份解析 ---
parsed_month = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")
bad_month_count = int(parsed_month.isna().sum())
if bad_month_count:
    raise ValueError("存在无法解析的月份, 需要复核。")
df["month"] = parsed_month
log("M03 无法解析月份行数", bad_month_count)

# --- 1.2 限定研究窗口 ---
df = df.loc[df["month"].between(STUDY_START, STUDY_END)].copy()
if df.empty:
    raise ValueError("研究期内没有记录。")
log("M03 研究窗口", "2022-07 至 2026-07")
log("M03 窗口内行数", len(df))

# --- 1.3 分析必需字段不得为空(month / make / fuel_type / vehicle_type) ---
analysis_required_key = ["month", "make", "fuel_type", "vehicle_type"]
missing_key_rows = df[analysis_required_key].eq("").any(axis=1) if "vehicle_type" in df else None
# 月份已转为 datetime, 这里只看文本字段
missing_key_rows = (df["make"].eq("") | df["fuel_type"].eq("") | df["vehicle_type"].eq(""))
missing_key_count = int(missing_key_rows.sum())
if missing_key_count:
    raise ValueError("分析必需字段仍有缺失, 需要复核。")
log("M03 关键字段缺失行数", missing_key_count)

# --- 1.4 数量: 空白按 0, 必须是非负整数 ---
blank_number_count = int(df["number"].eq("").sum())
parsed_number = pd.to_numeric(df["number"].replace("", "0"), errors="coerce")
bad_number_rows = parsed_number.isna() | parsed_number.lt(0) | parsed_number.mod(1).ne(0)
bad_number_count = int(bad_number_rows.sum())
if bad_number_count:
    raise ValueError("存在无法解析、负数或非整数的注册量。")
df["registrations"] = parsed_number.astype("int64")
log("M03 空白数量按0处理行数", blank_number_count)     # 2018-07 起 LTA 用空白表示零
log("M03 非法数量行数", bad_number_count)

# --- 1.5 原始重复记录检查(按 月/品牌/进口商/燃料/车身) ---
original_key = ["month", "make", "importer_type", "fuel_type", "vehicle_type"]
dup_rows = df.duplicated(subset=original_key, keep=False)
duplicate_row_count = int(dup_rows.sum())
if duplicate_row_count:
    raise ValueError("发现重复记录, 需要先确认原因。")
log("M03 原始重复记录行数", duplicate_row_count)

# --- 1.6 进口商维度检查: 未注明(空白)与已注明的正销量不得在同组并存 ---
importer_missing = df["importer_type"].eq("")
unspecified = df.loc[importer_missing & df["registrations"].gt(0), analysis_required_key].drop_duplicates()
specified   = df.loc[~importer_missing & df["registrations"].gt(0), analysis_required_key].drop_duplicates()
overlap = unspecified.merge(specified, on=analysis_required_key, how="inner")
overlap_count = len(overlap)
if overlap_count:
    raise ValueError("进口商未注明与已注明的记录在同一分组内并存, 需要复核。")
log("M03 进口商维度重叠组数", overlap_count)

# --- 1.7 车身类别标准化 ---
body_mapping = {
    "Sports Utility Vehicle": "SUV",
    "Sedan":                  "Sedan",
    "Multi-purpose Vehicle":  "MPV",
    "Hatchback":              "Hatchback",
    "Station-wagon":          "Station Wagon",
    "Coupe/ Convertible":     "Coupe/Convertible",
}
unknown_body_types = sorted(set(df["vehicle_type"]) - set(body_mapping))
if unknown_body_types:
    raise ValueError(f"存在未映射的车身类别: {unknown_body_types}")
df["body"] = df["vehicle_type"].map(body_mapping)
log("M03 车身类别映射", "SUV/Sedan/MPV/Hatchback/Station Wagon/Coupe-Convertible")

# --- 1.8 汇总到分析粒度: 月 x 品牌 x 燃料 x 车身(进口商类别加总消除) ---
fact_key = ["month", "make", "fuel_type", "body"]
total_before = int(df["registrations"].sum())
registration_fact = (df.groupby(fact_key, as_index=False)["registrations"]
                       .sum()
                       .sort_values(fact_key)
                       .reset_index(drop=True))
total_after = int(registration_fact["registrations"].sum())

fact_duplicate_count = int(registration_fact.duplicated(fact_key).sum())
target_bodies = ["SUV", "Sedan", "MPV"]
month_counts = (registration_fact.loc[registration_fact["body"].isin(target_bodies)]
                .groupby("body")["month"].nunique().reindex(target_bodies, fill_value=0))

if fact_duplicate_count: raise ValueError("最终粒度存在重复。")
if total_before != total_after: raise ValueError("汇总前后总量不一致。")
if not registration_fact["registrations"].ge(0).all(): raise ValueError("存在负注册量。")
if not month_counts.eq(49).all(): raise ValueError("目标车身月份不完整。")

log("M03 汇总前注册总量", total_before)
log("M03 汇总后注册总量", total_after)
log("M03 最终表行数", len(registration_fact))
log("M03 最终表重复组合数", fact_duplicate_count)
for b in target_bodies:
    log(f"M03 {b} 覆盖月份数", int(month_counts[b]))

# 月份导出为 YYYY-MM 文本, 便于 Excel 排序
export_fact = registration_fact.copy()
export_fact["month"] = export_fact["month"].dt.strftime("%Y-%m")
export_fact.to_csv(OUT / "registration_fact.csv", index=False, encoding="utf-8-sig")
print(f"  -> registration_fact.csv  行数 {len(export_fact)}")
print()

[1] 清洗 M03-Car_Regn_by_make.csv
  原始行数: 61359  列: ['month', 'make', 'importer_type', 'fuel_type', 'vehicle_type', 'number']
  M03 原始行数: 61359
  M03 源文件SHA256: dae4a01794d45a5a...
  M03 无法解析月份行数: 0
  M03 研究窗口: 2022-07 至 2026-07
  M03 窗口内行数: 33444
  M03 关键字段缺失行数: 0
  M03 空白数量按0处理行数: 25803
  M03 非法数量行数: 0
  M03 原始重复记录行数: 0
  M03 进口商维度重叠组数: 0
  M03 车身类别映射: SUV/Sedan/MPV/Hatchback/Station Wagon/Coupe-Convertible
  M03 汇总前注册总量: 172394
  M03 汇总后注册总量: 172394
  M03 最终表行数: 26898
  M03 最终表重复组合数: 0
  M03 SUV 覆盖月份数: 49
  M03 Sedan 覆盖月份数: 49
  M03 MPV 覆盖月份数: 49
  -> registration_fact.csv  行数 26898



In [3]:
# 2. COE 拥车证 —— 数值化(注意原文带千分位逗号)
# =============================================================================
print("[2] 清洗 COEBiddingResultsPrices.csv")
coe_path = find_raw("COEBiddingResultsPrices.csv")
coe = pd.read_csv(coe_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
coe.columns = coe.columns.str.strip()
for col in ["month", "bidding_no", "vehicle_class", "quota", "bids_success", "bids_received", "premium"]:
    coe[col] = coe[col].fillna("").astype(str).str.strip()

coe["month"] = pd.to_datetime(coe["month"], format="%Y-%m", errors="coerce")
if coe["month"].isna().any():
    raise ValueError("COE 月份存在异常。")

comma_cols = ["quota", "bids_success", "bids_received", "premium"]
comma_counts = {c: int(coe[c].str.contains(",").sum()) for c in comma_cols}
for c in comma_cols:                       # 去掉千分位逗号再转数字
    coe[c] = pd.to_numeric(coe[c].str.replace(",", "", regex=False), errors="coerce")
if coe[comma_cols].isna().any().any():
    raise ValueError("COE 数值列存在无法解析的值。")
coe["bidding_no"] = coe["bidding_no"].astype(int)

for c, n in comma_counts.items():
    log(f"COE {c} 含千分位逗号行数", n)
log("COE 行数", len(coe))
log("COE 月份范围", f"{coe['month'].min():%Y-%m} 至 {coe['month'].max():%Y-%m}")
coe.to_csv(OUT / "coe_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> coe_clean.csv  行数 {len(coe)}")
print()

[2] 清洗 COEBiddingResultsPrices.csv
  COE quota 含千分位逗号行数: 0
  COE bids_success 含千分位逗号行数: 41
  COE bids_received 含千分位逗号行数: 109
  COE premium 含千分位逗号行数: 0
  COE 行数: 1975
  COE 月份范围: 2010-01 至 2026-09
  -> coe_clean.csv  行数 1975



In [4]:
# 3. M09 车辆保有量
# =============================================================================
print("[3] 清洗 M09-Vehs_by_Fuel_Type.csv")
m09_path = find_raw("M09-Vehs_by_Fuel_Type.csv")
m09 = pd.read_csv(m09_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
m09.columns = m09.columns.str.strip()
for col in m09.columns:
    m09[col] = m09[col].fillna("").astype(str).str.strip()
m09["month"] = pd.to_datetime(m09["month"], format="%Y-%m", errors="coerce")
if m09["month"].isna().any():
    raise ValueError("M09 月份存在异常。")
# 源文件里 "-" 是占位符(表示缺失), 按 0 处理并记录数量
dash_count = int(m09["number"].eq("-").sum())
m09["number"] = m09["number"].replace({"": "0", "-": "0"})
m09["number"] = pd.to_numeric(m09["number"].str.replace(",", "", regex=False), errors="coerce")
if m09["number"].isna().any():
    raise ValueError("M09 数量存在无法解析的值。")
m09["number"] = m09["number"].astype("int64")
log("M09 占位符 '-' 按0处理行数", dash_count)
m09 = m09[["month", "category", "type", "number"]].sort_values(["month", "category", "type"]).reset_index(drop=True)
log("M09 行数", len(m09))
log("M09 月份范围", f"{m09['month'].min():%Y-%m} 至 {m09['month'].max():%Y-%m}")
m09.to_csv(OUT / "fleet_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> fleet_clean.csv  行数 {len(m09)}")
print()

[3] 清洗 M09-Vehs_by_Fuel_Type.csv
  M09 占位符 '-' 按0处理行数: 5
  M09 行数: 3611
  M09 月份范围: 2016-01 至 2026-07
  -> fleet_clean.csv  行数 3611



In [5]:
# 4. MVP02-2 年度新车注册
# =============================================================================
print("[4] 清洗 MVP02-2_New_Cars_by_make_type.csv")
mvp02_path = find_raw("MVP02-2_New_Cars_by_make_type.csv")
mvp02 = pd.read_csv(mvp02_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
mvp02.columns = mvp02.columns.str.strip()
for col in mvp02.columns:
    mvp02[col] = mvp02[col].fillna("").astype(str).str.strip()
mvp02["year"] = pd.to_numeric(mvp02["year"], errors="coerce").astype("Int64")
mvp02["number"] = mvp02["number"].replace({"": "0", "-": "0"})
mvp02["number"] = pd.to_numeric(mvp02["number"].str.replace(",", "", regex=False), errors="coerce").fillna(0).astype("int64")
mvp02 = mvp02[["year", "make", "importer_type", "fuel", "type", "number"]].sort_values(["year", "make", "fuel", "type"]).reset_index(drop=True)
log("MVP02-2 行数", len(mvp02))
log("MVP02-2 年份范围", f"{int(mvp02['year'].min())}-{int(mvp02['year'].max())}")
mvp02.to_csv(OUT / "mvp02_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> mvp02_clean.csv  行数 {len(mvp02)}")
print()

[4] 清洗 MVP02-2_New_Cars_by_make_type.csv
  MVP02-2 行数: 6896
  MVP02-2 年份范围: 2015-2025
  -> mvp02_clean.csv  行数 6896



In [6]:
# 5. MVP01-6 年度注册(按品牌)
# =============================================================================
print("[5] 清洗 MVP01-6_Cars_by_make.csv")
mvp01_path = find_raw("MVP01-6_Cars_by_make.csv")
mvp01 = pd.read_csv(mvp01_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
mvp01.columns = mvp01.columns.str.strip()
for col in mvp01.columns:
    mvp01[col] = mvp01[col].fillna("").astype(str).str.strip()
mvp01["year"] = pd.to_numeric(mvp01["year"], errors="coerce").astype("Int64")
mvp01["number"] = mvp01["number"].replace({"": "0", "-": "0"})
mvp01["number"] = pd.to_numeric(mvp01["number"].str.replace(",", "", regex=False), errors="coerce").fillna(0).astype("int64")
mvp01 = mvp01[["year", "make", "fuel_type", "number"]].sort_values(["year", "make"]).reset_index(drop=True)
log("MVP01-6 行数", len(mvp01))
log("MVP01-6 年份范围", f"{int(mvp01['year'].min())}-{int(mvp01['year'].max())}")
mvp01.to_csv(OUT / "mvp01_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> mvp01_clean.csv  行数 {len(mvp01)}")
print()

[5] 清洗 MVP01-6_Cars_by_make.csv
  MVP01-6 行数: 2198
  MVP01-6 年份范围: 2005-2025
  -> mvp01_clean.csv  行数 2198



In [7]:
# 6. 充电桩登记(列名标准化 + 日期/坐标数值化 + 去重)
# =============================================================================
print("[6] 清洗 Electric_Vehicle_Charging_Points_Jun 2026.csv")
ch_path = find_raw("Electric_Vehicle_Charging_Points_Jun 2026.csv")
ch = pd.read_csv(ch_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
ch.columns = ch.columns.str.strip()

rename = {
    "EV Charger Registration Code": "charger_code",
    "operator": "operator",
    "No. of Charging Outlets": "outlets",
    "evCpId": "cp_id",
    "plugType": "plug_type",
    "charging Speed": "charging_speed_kw",
    "PostalCode": "postal_code",
    "Block/House No": "block_no",
    "Street Name": "street_name",
    "Building Name": "building_name",
    "Floor No": "floor_no",
    "Lot No": "lot_no",
    "Is the charger publicly accessible?": "public_accessible",
    "longitude": "longitude",
    "latitude": "latitude",
    "Registration Date": "registration_date",
    "Type of Parking Lot": "parking_lot_type",
}
ch = ch.rename(columns=rename)
for col in ch.columns:
    ch[col] = ch[col].fillna("").astype(str).str.strip()

ch["outlets"] = pd.to_numeric(ch["outlets"].replace("", "0"), errors="coerce").fillna(0).astype(int)
ch["charging_speed_kw"] = pd.to_numeric(ch["charging_speed_kw"], errors="coerce")
ch["longitude"] = pd.to_numeric(ch["longitude"], errors="coerce")
ch["latitude"] = pd.to_numeric(ch["latitude"], errors="coerce")
ch["registration_date"] = pd.to_datetime(ch["registration_date"], errors="coerce", dayfirst=False)

before = len(ch)
ch = ch.drop_duplicates(subset=["cp_id"]).reset_index(drop=True)
log("充电桩 原始行数", before)
log("充电桩 去重后行数", len(ch))
log("充电桩 合计充电点(outlets)", int(ch["outlets"].sum()))
log("充电桩 登记日期范围", f"{ch['registration_date'].min():%Y-%m-%d} 至 {ch['registration_date'].max():%Y-%m-%d}")
ch.to_csv(OUT / "chargers_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> chargers_clean.csv  行数 {len(ch)}")
print()

[6] 清洗 Electric_Vehicle_Charging_Points_Jun 2026.csv
  充电桩 原始行数: 11353
  充电桩 去重后行数: 11353
  充电桩 合计充电点(outlets): 16935
  充电桩 登记日期范围: 2024-01-03 至 2026-12-04
  -> chargers_clean.csv  行数 11353

